# Level ROSETTA-ice gravity data

import the necessary packages

In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

import logging
import os

import geopandas as gpd
import numpy as np
import pandas as pd
import pooch
import verde as vd
from polartoolkit import maps
from polartoolkit import utils as polar_utils

from airbornegeo import levelling

os.environ["POLARTOOLKIT_HEMISPHERE"] = "south"

logging.getLogger("invert4geom").setLevel("WARN")
logging.basicConfig(level=logging.INFO)

## load data

In [ ]:
path = pooch.retrieve(
    url="http://wonder.ldeo.columbia.edu/data/ROSETTA-Ice/Gravity/rs_2019_grav.csv",
    fname="ROSETTA_2019_grav.csv",
    path=f"{pooch.os_cache('pooch')}/polartoolkit/gravity",
    known_hash="d4fdfcc293ac13e6222938bfa9e1e9ccc781ea7556a13d356e9f1b4aba809928",
    progressbar=True,
)
data_df = pd.read_csv(path)

# set standard column names
data_df = data_df.rename(
    columns={
        "Line": "line",
        "x": "easting",
        "y": "northing",
        "Height": "height",
        "FAG_levelled": "grav",
    }
)

# get rid of unneeded columns
data_df = data_df.drop(
    columns=[
        "LATITUDE",
        "LONGITUDE",
    ]
)

# convert line numbers into float format (L200 -> 200)
data_df.line = data_df.line.str[1:]
data_df["line"] = pd.to_numeric(data_df["line"])

# for simplicity, only keep a few lines
line = 840
data_df = data_df[
    data_df.line.isin(
        [
            # 450, 490, 650,
            # 530,
            line,
            # 1000, 1020, 1040, 1060, 1070, 1090, 1120,
            1120,
            1100,
            1090,
            1080,
        ]
    )
]

# convert from pandas dataframe to geopandas geodataframe
data_df = gpd.GeoDataFrame(
    data_df,
    geometry=gpd.points_from_xy(x=data_df.easting, y=data_df.northing),
    crs="EPSG:3031",
)

# calculate distance along flight lines
data_df["dist_along_line"] = levelling.distance_along_line(
    data_df,
    line_col_name="line",
    time_col_name="unixtime",
)

region = vd.pad_region(vd.get_region((data_df.easting, data_df.northing)), (20e3, 20e3))

data_df.head()

In [ ]:
# move 1 tie line up
# print(data_df[data_df.line == line].grav.mean())
# data_df.loc[data_df.line==line, "grav"] += 20
# print(data_df[data_df.line == line].grav.mean())

print(data_df[data_df.line == line].height.mean())
data_df.loc[data_df.line == line, "height"] += 500
print(data_df[data_df.line == line].height.mean())

In [ ]:
# save original gravity before un-levelling it
data_df["grav_original"] = data_df.grav

In [ ]:
# add a portion of a sin curve
df = data_df[data_df.line == line]

x = -df.dist_along_line
amplitude = 50
# period = (x.max()-x.min())*1.5
period = (x.max() - x.min()) * 2
data_df["curve"] = 0.0
curve = amplitude * np.sin(np.pi * x / period)
data_df.loc[data_df.line == line, "curve"] = curve - curve.mean()
data_df["grav"] = data_df.grav_original + data_df.curve

levelling.plot_line_and_crosses(
    data_df,
    line=line,
    x="dist_along_line",
    y=["grav_original", "grav", "curve"],
)

In [ ]:
# calculate theoretical intersection points
inters = levelling.create_intersection_table(
    flight_lines=data_df[data_df.line < 1000],
    tie_lines=data_df[data_df.line >= 1000],
    plot=False,
).reset_index(drop=True)


# interpolate data values at the intersection points
data_df, inters = levelling.interp1d_all_lines(
    data_df,
    inters,
    window_width=5e3,
    engine="scipy",
    method="cubic",
    extrapolate=True,
    fill_value="edge",
)
inters

In [ ]:
levelling.plot_line_and_crosses(
    data_df,
    line=line,
    x="dist_along_line",
    y=["grav"],
    plot_inters=True,
)

In [ ]:
inters = levelling.calculate_misties(
    inters,
    data_df,
    data_col="grav",
    plot=True,
)
inters

# Level without weights

In [ ]:
logging.getLogger("airbornegeo").setLevel("WARN")

# TREND 2
data_df, inters_levelled = levelling.iterative_line_levelling(
    inters,
    data_df,
    iterations=5,
    lines_to_level=data_df.line[data_df.line < 1000].unique(),
    degree=2,
    data_col="grav",
    levelled_col="grav_trend_2",
    # plot_convergence=True,
)

In [ ]:
data_df["levelling_corr"] = data_df.grav - data_df.grav_trend_2
levelling.plot_line_and_crosses(
    data_df,
    line=line,
    x="dist_along_line",
    y=[
        # "grav",
        "grav_original",
        "grav_trend_2",
        # "levelling_corr",
    ],
    # y_axes=[1,1,2],
    plot_inters=[
        # False,
        False,
        True,
        # False
    ],
)

In [ ]:
cpt_lims = polar_utils.get_min_max(
    inters_levelled.mistie_0,
    robust=False,
    absolute=True,
)
fig = maps.basemap(
    region=region,
    title="Unlevelled misties",
)
levelling.plot_flightlines(
    fig, data_df[data_df.line < 1000], plot_lines=True, direction="EW"
)
levelling.plot_flightlines(
    fig, data_df[data_df.line >= 1000], plot_lines=True, direction="NS"
)

fig = maps.basemap(
    fig=fig,
    frame=None,
    origin_shift=None,
    points=inters_levelled,
    points_cmap="balance+h0",
    points_style="c10p",
    points_pen="black",
    points_fill=inters_levelled.mistie_0,
    cbar_label="nT",
    hist=True,
    # robust=True,
    cpt_lims=cpt_lims,
)

fig = maps.basemap(
    fig=fig,
    origin_shift="x",
    title="Post-trend 2 misties",
)
levelling.plot_flightlines(
    fig, data_df[data_df.line < 1000], plot_lines=True, direction="EW"
)
levelling.plot_flightlines(
    fig, data_df[data_df.line >= 1000], plot_lines=True, direction="NS"
)

fig = maps.basemap(
    fig=fig,
    frame=None,
    origin_shift=None,
    points=inters_levelled,
    points_cmap="balance+h0",
    points_style="c10p",
    points_pen="black",
    points_fill=inters_levelled.mistie,
    cbar_label="nT",
    hist=True,
    # robust=True,
    cpt_lims=cpt_lims,
)

fig.show()

# Level with manual intersection weights

In [ ]:
# set weight for intersection with 1040 to be lower than the rest
inters["mistie_weight"] = np.where(inters.tie == 1100, 0.001, 1)
inters

In [ ]:
fig = maps.basemap(
    region=region,
    title="Intersection weights",
)
levelling.plot_flightlines(fig, data_df[data_df.line < 1000], direction="EW")
levelling.plot_flightlines(fig, data_df[data_df.line >= 1000], direction="NS")

fig = maps.basemap(
    fig=fig,
    frame=None,
    origin_shift=None,
    points=inters,
    points_cmap="matter",
    reverse_cpt=True,
    points_style="c10p",
    points_pen="black",
    points_fill=inters.mistie_weight,
    cbar_label="weight factor",
    hist=True,
    cpt_lims=(0, 1),
)

fig.show()

In [ ]:
logging.getLogger("airbornegeo").setLevel("WARN")

# TREND 2
data_df, inters_levelled = levelling.iterative_line_levelling(
    inters,
    data_df,
    iterations=5,
    lines_to_level=data_df.line[data_df.line < 1000].unique(),
    degree=2,
    data_col="grav",
    levelled_col="grav_trend_2_weights",
    sample_weight_col="mistie_weight",
    # plot_convergence=True,
)

In [ ]:
data_df["levelling_corr"] = data_df.grav - data_df.grav_trend_2
levelling.plot_line_and_crosses(
    data_df,
    line=line,
    x="dist_along_line",
    y=[
        # "grav",
        "grav_original",
        "grav_trend_2",
        "grav_trend_2_weights",
        # "levelling_corr",
    ],
    # y_axes=[1,1,2],
    plot_inters=[
        # False,
        False,
        True,
        False,
        # False
    ],
)

# Level with automatic intersection weights

In [ ]:
data_df["grav_filt"] = levelling.filter_flight_lines(
    data_df,
    filt_type="g20000",
    data_column="grav",
)
data_df["grav_1st_derive"] = np.abs(
    data_df.groupby("line")
    .apply(
        lambda x: pd.Series(np.gradient(x.grav_filt, x.dist_along_line), index=x.index),
        include_groups=False,
    )
    .reset_index(drop=True)
)
data_df["grav_1st_derive"] = np.abs(
    levelling.filter_flight_lines(
        data_df,
        filt_type="g20000",
        data_column="grav_1st_derive",
    )
)

data_df["grav_2nd_derive"] = np.abs(
    data_df.groupby("line")
    .apply(
        lambda x: pd.Series(
            np.gradient(x.grav_1st_derive, x.dist_along_line), index=x.index
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)
data_df["grav_2nd_derive"] = np.abs(
    levelling.filter_flight_lines(
        data_df,
        filt_type="g20000",
        data_column="grav_2nd_derive",
    )
)
data_df.head()

In [ ]:
levelling.plot_line_and_crosses(
    data_df,
    line=line,
    x="dist_along_line",
    y=[
        "grav",
        "grav_1st_derive",
        "grav_2nd_derive",
    ],
    y_axes=[
        1,
        2,
        3,
    ],
    # plot_inters=[False, True, True],
)

In [ ]:
# data_df["grav_filt"] = levelling.filter_flight_lines(
#     data_df,
#     filt_type="g20000",
#     data_column="grav",
# )
data_df["height_1st_derive"] = np.abs(
    data_df.groupby("line")
    .apply(
        lambda x: pd.Series(np.gradient(x.height, x.dist_along_line), index=x.index),
        include_groups=False,
    )
    .reset_index(drop=True)
)
data_df["height_1st_derive"] = np.abs(
    levelling.filter_flight_lines(
        data_df,
        filt_type="g20000",
        data_column="height_1st_derive",
    )
)

data_df["height_2nd_derive"] = np.abs(
    data_df.groupby("line")
    .apply(
        lambda x: pd.Series(
            np.gradient(x.height_1st_derive, x.dist_along_line), index=x.index
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)
data_df["height_2nd_derive"] = np.abs(
    levelling.filter_flight_lines(
        data_df,
        filt_type="g20000",
        data_column="height_2nd_derive",
    )
)
data_df.head()

In [ ]:
levelling.plot_line_and_crosses(
    data_df,
    line=line,
    x="dist_along_line",
    y=[
        "height",
        "height_1st_derive",
        "height_2nd_derive",
    ],
    y_axes=[
        1,
        2,
        3,
    ],
    plot_inters=[False, True, True],
    # use_intersection_y=False,
)

### Create weights for each intersection
When fitting polynomials to each line's mistie values, we can weight certain intersection more or less. This may depend on several factors, such as: 
* distance of intersection to actual data
* if the intersection was interpolated or extrapolated
* if the intersection is at a location of high gradient magnetic data or flight elevations
* if there is a large elevation difference between the flight and the tie at the intersection

We convert these factors into numeric values, where higher values mean the intersection is reliable. 

Should these weights be calculated based on percentile rank for the entire dataset, or just per line? 

In [ ]:
inters = levelling.calculate_intersection_weights(
    inters,
    data_df,
    weight_by="all",
    max_dist_weight=1,
    height_difference_weight=1,
    interpolation_type_weight=1,
    data_1st_derive_weight=1,
    data_1st_derive_col_name="grav_1st_derive",
    data_2nd_derive_weight=1,
    data_2nd_derive_col_name="grav_2nd_derive",
    height_1st_derive_weight=1,
    height_1st_derive_col_name="height_1st_derive",
    height_2nd_derive_weight=1,
    height_2nd_derive_col_name="height_2nd_derive",
    # plot=True,
)
inters.sort_values(by="mistie_weight", ascending=False)

In [ ]:
fig = maps.basemap(
    region=region,
    title="Intersection weights",
)
levelling.plot_flightlines(fig, data_df[data_df.line < 1000], direction="EW")
levelling.plot_flightlines(fig, data_df[data_df.line >= 1000], direction="NS")

fig = maps.basemap(
    fig=fig,
    frame=None,
    origin_shift=None,
    points=inters,
    points_cmap="matter",
    reverse_cpt=True,
    points_style="c10p",
    points_pen="black",
    points_fill=inters.mistie_weight,
    cbar_label="weight factor",
    hist=True,
    cpt_lims=(0, 1),
)

fig.show()

In [ ]:
logging.getLogger("airbornegeo").setLevel("WARN")

# TREND 2
data_df, inters_levelled = levelling.iterative_line_levelling(
    inters,
    data_df,
    iterations=5,
    lines_to_level=data_df.line[data_df.line < 1000].unique(),
    degree=2,
    data_col="grav",
    levelled_col="grav_trend_2_auto_weights",
    sample_weight_col="mistie_weight",
    # plot_convergence=True,
)

In [ ]:
data_df["levelling_corr"] = data_df.grav - data_df.grav_trend_2
levelling.plot_line_and_crosses(
    data_df,
    line=line,
    x="dist_along_line",
    y=[
        # "grav",
        "grav_original",
        "grav_trend_2",
        "grav_trend_2_weights",
        "grav_trend_2_auto_weights",
        # "levelling_corr",
    ],
    # y_axes=[1,1,2],
    plot_inters=[
        # False,
        False,
        True,
        False,
        False,
        # False
    ],
)